In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
DATA = Path("data")

## The data: MNIST 0s and 1s, shrunk to 6 x 6
Target y = 1 for a "1", y = 0 for a "0". Shrinking keeps the shapes as small as in the text.

In [ ]:
(x_train, y_train), _ = keras.datasets.mnist.load_data()
keep = np.isin(y_train, [0, 1])
small = np.array([np.array(Image.fromarray(im).resize((6, 6), Image.BILINEAR)) for im in x_train[keep][:64]]) / 255.0
y = (y_train[keep][:64] == 1).astype(float)
print(small.shape, "first targets:", y[:8])
print(small[0].round(2))

## Forward propagation, one image

In [ ]:
rng = np.random.default_rng(0)
W1 = rng.normal(0, 0.5, (3, 3)); b1 = 0.1             # the filter and its bias
W2 = rng.normal(0, 0.5, (1, 4)); b2 = 0.0             # the output node's 4 weights and its bias

def conv2d(X, K):
    n = X.shape[0] - K.shape[0] + 1
    return np.array([[(X[i:i + K.shape[0], j:j + K.shape[1]] * K).sum() for j in range(n)] for i in range(n)])

def maxpool(A):
    return A.reshape(2, 2, 2, 2).max(axis=(1, 3))     # 4 x 4 -> 2 x 2, windows 2 x 2, stride 2

def forward(X):
    Z1 = conv2d(X, W1) + b1
    A1 = np.maximum(Z1, 0)
    P1 = maxpool(A1)
    F = P1.reshape(4, 1)                               # flatten to a column
    Z2 = W2 @ F + b2
    A2 = 1 / (1 + np.exp(-Z2))
    return Z1, A1, P1, F, Z2, A2

X, t = small[0], y[0]
Z1, A1, P1, F, Z2, A2 = forward(X)
for name, v in zip(("X", "Z1", "A1", "P1", "F", "Z2", "A2"), (X, Z1, A1, P1, F, Z2, A2)):
    print(f"{name:3s} shape {v.shape}")
loss = -(t * np.log(A2) + (1 - t) * np.log(1 - A2))
print("prediction", A2.item(), " target", t, " loss", loss.item())
print("trainable parameters:", W1.size + 1 + W2.size + 1)

## The last layer's gradients by hand: dL/dW2 = (a2 - y) F^T, dL/db2 = a2 - y

In [ ]:
dW2 = (A2 - t) @ F.T
print("F^T =", F.T.round(4), " a2 - y =", (A2 - t).round(4))
db2 = (A2 - t).item()
print("dL/dW2 =", dW2.round(5), " shape", dW2.shape)
print("dL/db2 =", round(db2, 5))

## The same gradients from TensorFlow

In [ ]:
def tf_forward(X, W1, b1, W2, b2):
    Z1 = tf.nn.conv2d(X[None, :, :, None], W1[:, :, None, None], 1, "VALID")[0, :, :, 0] + b1
    P1 = tf.nn.max_pool2d(tf.nn.relu(Z1)[None, :, :, None], 2, 2, "VALID")[0, :, :, 0]
    return tf.sigmoid(W2 @ tf.reshape(P1, (4, 1)) + b2)

v = [tf.Variable(np.array(a, float)) for a in (W1, b1, W2, b2)]
with tf.GradientTape() as tape:
    a2 = tf_forward(tf.constant(X), *v)
    L = -(t * tf.math.log(a2) + (1 - t) * tf.math.log(1 - a2))
g = tape.gradient(L, v)
print("TensorFlow dL/dW2 =", g[2].numpy().round(5))
print("TensorFlow dL/db2 =", g[3].numpy().round(5).ravel())
print("largest difference:", max(np.abs(g[2].numpy() - dW2).max(), abs(g[3].numpy().item() - db2)))

## A batch of m images: F is 4 x m, the gradient stays 1 x 4
With the loss averaged over the batch, a 1/m appears: dL/dW2 = (1/m) (A2 - Y) F^T.

In [ ]:
m = 32
Fs = np.hstack([forward(small[i])[3] for i in range(m)])           # 4 x m
A2s = 1 / (1 + np.exp(-(W2 @ Fs + b2)))                               # 1 x m
Y = y[:m].reshape(1, m)
dW2_batch = (A2s - Y) @ Fs.T / m
print("F", Fs.shape, " A2", A2s.shape, " dL/dW2", dW2_batch.shape)
with tf.GradientTape() as tape:
    preds = tf.concat([tf_forward(tf.constant(small[i]), *v) for i in range(m)], axis=1)
    L = tf.reduce_mean(-(Y * tf.math.log(preds) + (1 - Y) * tf.math.log(1 - preds)))
g = tape.gradient(L, v)
print("largest difference from TensorFlow:", np.abs(g[2].numpy() - dW2_batch).max(),
      abs(g[3].numpy().item() - (A2s - Y).mean()))
pd.DataFrame({"name": ["dW2_1", "dW2_2", "dW2_3", "dW2_4", "db2"], "value": list(dW2.ravel()) + [db2]}).to_csv(
    DATA / "last_layer_grads.csv", index=False)